## CS 5970: Assignment 2 - How do LLMs learn?
______________________________________________

**Professor:** Dr. Sathyanarayanan (Sathya) Aakur

**Teaching Assistant:** Chenjia Li

**Student:** Hannah Schmucker

**03/06/2026**

Overview: Implement a GPT-style Transformer language model from configuration, pretrain, then perform supervised fine-tuning (SFT) and analyze changes in both behavior and representations.

Focus:
- understanding casual language modeling
- implementing response-only SFT
- analyzing representation shift and forgetting
- comparing decoding strategies

**Stage 1: Pretraining (Casual LM)**

In [15]:
# Install dependencies
!pip -q install sentencepiece datasets transformers accelerate

# Import packages
import os, math, urllib.request, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
from transformers import AutoConfig, AutoModelForCausalLM, GPT2Config

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [16]:
# for some reason the dataset wasn't loading. I referenced the link below
#     which suggested downloading the ds using an older version
# https://github.com/huggingface/datasets/issues/7693
!pip install datasets==3.6.0

# (b) Get data
from datasets import load_dataset
dataset = load_dataset("ptb_text_only",download_mode="force_redownload",revision="main")

train_ds = dataset["train"]
val_ds = dataset["validation"]
test_ds = dataset["test"]

text = "\n".join(train_ds["sentence"] + val_ds["sentence"] + test_ds["sentence"])

print ("Character count: ", len(text))
print("Training dataset example:", train_ds[:5])
print("Validation dataset example:", val_ds[:5])
print("Test dataset example:", test_ds[:5])
print("Text example:", text[:100])


ptb_text_only.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/42068 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3761 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3370 [00:00<?, ? examples/s]

Character count:  5852946
Training dataset example: {'sentence': ['aer banknote berlitz calloway centrust cluett fromstein gitano guterman hydro-quebec ipo kia memotec mlx nahb punts rake regatta rubens sim snack-food ssangyong swapo wachter', 'pierre <unk> N years old will join the board as a nonexecutive director nov. N', 'mr. <unk> is chairman of <unk> n.v. the dutch publishing group', 'rudolph <unk> N years old and former chairman of consolidated gold fields plc was named a nonexecutive director of this british industrial conglomerate', 'a form of asbestos once used to make kent cigarette filters has caused a high percentage of cancer deaths among a group of workers exposed to it more than N years ago researchers reported']}
Validation dataset example: {'sentence': ['consumers may want to move their telephones a little closer to the tv set', "<unk> <unk> watching abc 's monday night football can now vote during <unk> for the greatest play in N years from among four or five <unk> <u

In [17]:
# (c) Train SentencePiece tokenizer on ptb corpus
import sentencepiece as spm

sp_dir = "spm_gpt"
os.makedirs(sp_dir, exist_ok=True)
corpus_path = os.path.join(sp_dir, "corpus.txt")

with open(corpus_path, "w", encoding="utf-8") as f:
    f.write(text)

spm_prefix = os.path.join(sp_dir, "ptb_spmtokenizer")
vocab_size = 8000

spm.SentencePieceTrainer.Train(
    input=corpus_path,
    model_prefix=spm_prefix,
    vocab_size=vocab_size,
    model_type="bpe",
    character_coverage=1.0,
    bos_id=-1, eos_id=-1, pad_id=0, unk_id=1,  # pad/unk fixed
    user_defined_symbols=["<extra_id_0>","<extra_id_1>","<extra_id_2>","<extra_id_3>","<extra_id_4>"]
)

sp = spm.SentencePieceProcessor()
sp.load(spm_prefix + ".model")
print("SP vocab size: ", sp.get_piece_size())

# Tokenize
tokens = sp.encode(text)
print("Total tokens:", len(tokens))

# Convert to tensor
data = torch.tensor(tokens, dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
validation_data = data[n:]

print("Training data size: ", len(train_data))
print("Validation data size: ", len(validation_data))


SP vocab size:  8000
Total tokens: 1324938
Training data size:  1192444
Validation data size:  132494


In [18]:
# (d) Initialize the AutoConfig (GPT2 model) from HuggingFace [not pretrained]

d_model = 256
d_ff = 1024

config = GPT2Config(
    vocab_size=8000,
    n_positions=128,
    n_embd=d_model,
    n_layer=4,
    n_head=4,
    n_inner=d_ff
# Dropout and LayerNorm default at 0.1 and pre-LN (1e-05)
# for this (GPT) model.
# https://huggingface.co/docs/transformers/v5.3.0/en/model_doc/gpt2#transformers.GPT2Config

)

In [19]:
# (f) Implement a dataloader
block_size = 128
batch_size = 64

def get_batch(split="train"):
    d = train_data if split=="train" else validation_data

    # Rand samples contiguous blocks of length 128
    ix = torch.randint(0, len(d)-block_size-1, (batch_size,))

    # Uses proper shifting
    input_ids = torch.stack([d[i:i+block_size] for i in ix])
    labels = torch.stack([d[i+1:i+block_size+1] for i in ix])

    # Produces input_ids and label tensors
    return input_ids.to(device),labels.to(device)

# loss eval
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}

    for split in ["train","val"]:
        losses = []

        for _ in range(50):
            x,y = get_batch(split)
            outputs = model(input_ids=x,labels=y)
            loss = outputs.loss

            losses.append(loss.item())

        out[split] = sum(losses)/len(losses)

    model.train()
    return out


# (e) Train as a causal language model

model = AutoModelForCausalLM.from_config(config)
torch.save(model.state_dict(), "sanitycheck.pt")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# (g) train for 3,000 optimization steps
steps = 3000

train_losses = []
validation_losses = []
step_list = []

for step in range(steps):
  x,y = get_batch("train")

  outputs = model(input_ids=x,labels=y)
  loss = outputs.loss

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if step % 200 == 0:
    losses = estimate_loss()

    print(f"Step {step}")
    print(f"Training loss: {losses["train"]:.4f}")
    print(f"Validation loss: {losses["val"]:.4f}")

    train_losses.append(losses["train"])
    validation_losses.append(losses["val"])
    step_list.append(step)


# Save the model (now trained) for the SFT task
torch.save(model.state_dict(), "pretrained.pt")


Step 0
Training loss: 8.7620
Validation loss: 8.7381


KeyboardInterrupt: 

In [ ]:
# (h) Report metrics

final_losses = estimate_loss()
train_loss = final_losses["train"]
val_loss = final_losses["val"]
val_perplexity = torch.exp(torch.tensor(val_loss))

print("===REPORT===")
print("Training loss: ", train_loss)
print("Validation loss: ", val_loss)
print("Validation perplexity: ", val_perplexity.item())

import matplotlib.pyplot as plt
plt.plot(train_losses, label="Training loss")
plt.plot(validation_losses, label="Validation loss")

plt.xlabel("Steps")
# Higher C.E. loss = bad
plt.ylabel("Cross Entropy Loss")
plt.title("Train vs Validation Loss")
plt.grid(True)
plt.legend()
plt.show()

**Stage 2: SFT**

In [ ]:
# === (a) Use a new dataset ===
ds = load_dataset("databricks/databricks-dolly-15k", split="train").shuffle(seed=42).select(range(2000))
ds_small = ds.shuffle(seed=42).select(range(200))

  # 90/10 split for 2000 SFT examples
data = ds.to_list()
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

# === (b) format ===
def format_dolly(ex):
    inst = (ex.get("instruction") or "").strip()
    resp = (ex.get("response") or "").strip()
    prompt = f"### Instruction:\n{inst}\n\n### Response:\n"
    text = prompt + resp
    return {"prompt": prompt, "text": text}

ds = ds.map(format_dolly, remove_columns=ds.column_names)
splits = ds.train_test_split(test_size=0.1, seed=42)
train_sft, val_sft = splits["train"], splits["test"]
len(train_sft), len(val_sft)


# === (c) Tokenize using previous tokenizer (I used a large part of the demo here) ===
max_len = 256

def tokenize_sft_with_spt(ex):
  prompt = ex["prompt"]
  # tokenize prompt (sp. is the spt defined in part 1)
  prompt_ids = sp.encode(prompt, out_type = int)
  prompt_len = len(prompt_ids)

  text = ex["text"]
   # tokenize text
  text_ids = sp.encode(text,out_type = int)
  input_ids = text_ids

   # prevent the model from 'cheating', fill all with 1s
  attention_mask = [1] * len(input_ids)

  labels = [-100] * len(input_ids)

  # (d) Implement response-only loss masking
  # don't use prompt tokens; start from the end of the
  #     instruction and ready to the end of the response
  for i in range(prompt_len, len(input_ids)):
    labels[i] = input_ids[i]

  return {
      "input_ids": input_ids,
      "labels": labels,
      "attention_mask":attention_mask,
      "prompt": ex["prompt"]
  }

 # Tokenize the SFT training/validation sets
train_tok = train_sft.map(tokenize_sft_with_spt)
val_tok = val_sft.map(tokenize_sft_with_spt)

  # checking size real quick
print ("Training length: ", len(train_data))
print ("Validation length: ", len(val_data))



def pad_seq(sequences, pad_value, max_len=None):
  if max_len is None:
    max_len = max(len(s) for s in sequences)

  padded = []
  for s in sequences:
    s = list(s)
    if len(s) < max_len:
      s = s + [pad_value] * (max_len - len(s))
    else:
      s = s[:max_len]

    padded.append(s)
  return torch.tensor(padded, dtype=torch.long)

 # Collate
def collate(features):
  input_ids = [f["input_ids"] for f in features]
  attention_masks = [f["attention_mask"] for f in features]
  labels = [f["labels"] for f in features]
  prompts = [f["prompt"] for f in features]

  # Found eos_id()
  max_len = max(len(x) for x in input_ids)
  input_ids = pad_seq(input_ids, pad_value=sp.eos_id(), max_len=max_len).to(device)
  attention_masks = pad_seq(attention_masks, pad_value=0, max_len=max_len).to(device)
  labels = pad_seq(labels, pad_value=-100, max_len=max_len).to(device)

  return {
      "input_ids": input_ids,
      "attention_mask": attention_masks,
      "labels": labels,
      "prompt": prompts
  }


In [ ]:
from transformers import TrainingArguments, Trainer

# Initialize the new SFT model
sft_model = AutoModelForCausalLM.from_config(config)
sft_model.load_state_dict(torch.load("pretrained.pt", map_location=device))
sft_model.config.pad_token_id = 0
sft_model.to(device)
 # sanity check
print(sp.vocab_size())
print(sft_model.config.pad_token_id)

sft_sm_model = AutoModelForCausalLM.from_config(config)
sft_sm_model.load_state_dict(torch.load("pretrained.pt", map_location=device))
sft_sm_model.config.pad_token_id = 0
sft_sm_model.to(device)

args = TrainingArguments(
    output_dir="sft_model",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch ~32
    num_train_epochs=10,
    learning_rate=5e-5,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,                  # ensures we save at least once
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=sft_model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collate
)

args_small = TrainingArguments(
    output_dir="sft_sm_model",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch ~32
    num_train_epochs=10,
    learning_rate=5e-5,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,                  # ensures we save at least once
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none"
)

train_smaller_tok = train_tok.shuffle(seed=42).select(range(200))
trainer_sm = Trainer (
    model=sft_sm_model,
    args=args,
    train_dataset=train_smaller_tok,
    eval_dataset=val_tok,
    data_collator=collate
)


# Train and save with 200 SFT examples
trainer_sm.train()
trainer_sm.save_model("sft_sm_model")

trainer.train()
trainer.save_model("sft_model")


In [ ]:
# (f) Generate text using the same 6 points

eval_prompts = [
    "### Instruction:\nSummarize in one sentence: Neural networks learn representations from data.\n\n### Response:\n",
    "### Instruction:\nGive 3 bullet points about overfitting.\n\n### Response:\n",
    "### Instruction:\nTranslate to French: hello world\n\n### Response:\n",
    "### Instruction:\nClassify sentiment as positive or negative: I hated this movie.\n\n### Response:\n",
    "### Instruction:\nRewrite in passive voice: The dog chased the cat.\n\n### Response:\n",
    "### Instruction:\nExtract JSON with keys name and age: John is 25 years old.\n\n### Response:\n",
]

@torch.no_grad()
def generate_one(model, prompt, max_new_tokens=80, do_sample=True, temperature=0.8, top_p=0.9, num_beams=1):
    model.eval()
    x = tok(prompt, return_tensors="pt").to(device)
    y = model.generate(
        **x,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    return tok.decode(y[0], skip_special_tokens=True)

def eval_model(model, name):
    gens = [generate_one(model, p) for p in eval_prompts]
    r = reward_fn(gens)
    print(f"\n==== {name} ====")
    for i,(g,rr) in enumerate(zip(gens, r)):
        print("-"*80)
        print(f"Prompt {i+1} reward={rr:.3f}")
        print(g)
    print("-"*80)
    mean_r = sum(r)/len(r)
    print(f"{name}: mean reward = {mean_r:.3f}")
    return mean_r

# After random initialization
eval_model(model, "Random")
# After pretraining

# After SFT

# Greedy decoding
# Top-p sampling (p=0.9)
# Report

**Stage 3: Representation Analysis**

**Stage 4: Insight Blocks**

1: Using Eval 3, did SFT increase PTB perplexity? Why would this happen?

2: Using Eval 1 or Eval 2, describe one measurable cahnge in representation between pretraining and SFT.

3: Compare greedy vs top-p outputs. Which better reflects instruction-following behavior and why?

4: Describe you ablation (freeze or data-size). What changed quantitatively?